# Compression: trade-offs between speed and size

In this demo and exercise, we explore the trade-off between reading and writing speed, and the size of the data on disk. Again, we use the two-photon data we wrote to a Zarr file on Monday.

We start by lazily opening the two-photon data (in read-only mode, `mode="r"`) with `zarr-python`, which allows us to inspect the compression information. 

In [ ]:
from pathlib import Path
import zarr

in_path = Path("./data/twophoton_series.zarr")

two_photon_series_zarr = zarr.open(in_path, mode='r')
print(two_photon_series_zarr.info)

Because we wrote this earlier in the course, we just used the default zarr compression: a Zstd Codec with compression level 0. We know want to write this same array back to disk, but with the Blosc Codec and varying compression levels. We start by reading the same array into `dask`, because this will allow us to easily copy it chunk-by-chunk

In [ ]:
import dask.array as da
two_photon_series = da.from_zarr(in_path, mode='r')

In [46]:
def zarr_disk_size(path):
    return sum(
        f.stat().st_size
        for f in Path(path).rglob("*")
        if f.is_file()
    )

Next, we loop over different compessions levels (0,3,6), and for each compression level, we

1. open a Zarr file on disk, specifying the compressor we want
2. write the array to disk chunk by chunk
3. record how much time it took
4. record how big the Zarr file is on disk

In [ ]:
import time
from zarr.codecs import BloscCodec

write_times = []
sizes_on_disk = []
compression_levels = range(0,9,3)

for clevel in compression_levels:
    start = time.time()

    # step 1.
    outpath = Path(f"./data/two_photon_series_clevel-{clevel}.zarr")
    destination_on_disk = zarr.create_array(
        outpath,
        shape=two_photon_series.shape,
        dtype=two_photon_series.dtype,
        chunks=two_photon_series.chunksize,
        zarr_format=3,
        overwrite=True,
        compressors=[
            BloscCodec(cname="zstd", clevel=clevel),
        ],
    )

    # step 2.
    two_photon_series.to_zarr(
        destination_on_disk,
        compute=True,
        mode="w",
    )

    # step 1.
    stop = time.time()
    print(f"Elapsed: {stop-start}")
    write_times.append(stop-start)
    sizes_on_disk.append(zarr_disk_size(outpath))

Elapsed: 0.9059271812438965


/Users/alessandrofelder/dev/slides-large-array-data-osss-2026/.venv/lib/python3.14/site-packages/dask/array/core.py:3015: PerformanceWarning: The input Dask array will be rechunked along axis 1 with chunk size 248, but a chunk size divisible by 14 is required for Dask to write safely to the Zarr array <Array file://data/two_photon_series_clevel-3.zarr shape=(500, 248, 440) dtype=int16>. To avoid risk of data loss when writing to this Zarr array, set the "array.chunk-size" configuration parameter to at least the size in bytes of a single on-disk chunk (or shard) of the Zarr array, which in this case is 196000 bytes. E.g., dask.config.set({"array.chunk-size": 196000})
  return to_zarr(self, *args, **kwargs)


Elapsed: 1.6278636455535889


/Users/alessandrofelder/dev/slides-large-array-data-osss-2026/.venv/lib/python3.14/site-packages/dask/array/core.py:3015: PerformanceWarning: The input Dask array will be rechunked along axis 1 with chunk size 248, but a chunk size divisible by 14 is required for Dask to write safely to the Zarr array <Array file://data/two_photon_series_clevel-6.zarr shape=(500, 248, 440) dtype=int16>. To avoid risk of data loss when writing to this Zarr array, set the "array.chunk-size" configuration parameter to at least the size in bytes of a single on-disk chunk (or shard) of the Zarr array, which in this case is 196000 bytes. E.g., dask.config.set({"array.chunk-size": 196000})
  return to_zarr(self, *args, **kwargs)


Elapsed: 3.315223217010498


We can now use our results to plot the trade-off between size-on-disk and writing speed.

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.plot()
plt.show()